## Setup
This notebook requires a DeepSeek API key (free tier works).
Get one at: https://platform.deepseek.com
If running locally, add DEEPSEEK_API_KEY to a .env file.
If running on Colab, you will be prompted to enter it.

In [ ]:
import sys
!{sys.executable} -m pip install orchestral-ai pennylane torch torchvision python-dotenv

In [ ]:
import pennylane as qml
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from orchestral import Agent, define_tool
from orchestral.llm import VLLM
from dotenv import load_dotenv
import numpy as np
import os

load_dotenv()

api_key = os.getenv('DEEPSEEK_API_KEY')
if not api_key:
    import getpass
    api_key = getpass.getpass("Enter DeepSeek API key: ")

In [ ]:
@define_tool()
def hilbert_space_dimension(n_qubits: int) -> str:
    """Returns the dimension of the Hilbert space for an n-qubit system.

    Args:
        n_qubits: Number of qubits in the system
    Returns:
        Dimension of the Hilbert space (2^n_qubits) as a string
    """
    dim = 2 ** n_qubits
    return f"A {n_qubits}-qubit system has a Hilbert space of dimension {dim}"

agent = Agent(
    llm=VLLM(model='deepseek-chat', host='https://api.deepseek.com', api_key=api_key),
    tools=[hilbert_space_dimension],
    system_prompt="You are a quantum computing assistant. Use tools when asked about quantum systems."
)

response = agent.run("What is the dimension of the Hilbert space for a 4-qubit system?")
print(response.text)

In [ ]:
def build_circuit(n_qubits, n_layers, entanglement_type, embedding_type):
    dev = qml.device("default.qubit", wires=n_qubits)

    @qml.qnode(dev, interface="torch")
    def circuit(inputs, weights):
        if embedding_type == "angle":
            qml.AngleEmbedding(inputs, wires=range(n_qubits))
        elif embedding_type == "amplitude":
            qml.AmplitudeEmbedding(inputs, wires=range(n_qubits), normalize=True)
        for _ in range(n_layers):
            for i in range(n_qubits):
                qml.RY(weights[_, i], wires=i)
            if entanglement_type == "linear":
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i+1])
            elif entanglement_type == "circular":
                for i in range(n_qubits):
                    qml.CNOT(wires=[i, (i+1) % n_qubits])
            elif entanglement_type == "full":
                for i in range(n_qubits):
                    for j in range(i+1, n_qubits):
                        qml.CNOT(wires=[i, j])
        return qml.expval(qml.PauliZ(0))

    return circuit

In [ ]:
class HybridQNN(nn.Module):
    def __init__(self, n_qubits, n_layers, entanglement_type, embedding_type):
        super().__init__()
        self.pre    = nn.Linear(784, n_qubits)
        self.weights = nn.Parameter(torch.randn(n_layers, n_qubits))
        self.post   = nn.Linear(1, 10)
        self.circuit = build_circuit(n_qubits, n_layers, entanglement_type, embedding_type)

    def forward(self, x):
        x = torch.sigmoid(self.pre(x.view(-1, 784)))
        x = torch.stack([self.circuit(x[i], self.weights)
                         for i in range(len(x))]).float()
        return self.post(x.unsqueeze(1))

In [ ]:
transform    = transforms.Compose([transforms.ToTensor()])
full_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
subset       = Subset(full_dataset, range(512))
loader       = DataLoader(subset, batch_size=32, shuffle=True)

In [ ]:
import pandas as pd
search_results = []

@define_tool()
def train_qnn(
    epochs: int,
    learning_rate: float,
    n_layers: int = 2,
    entanglement_type: str = "linear",
    embedding_type: str = "angle"
) -> str:
    """Trains a hybrid quantum-classical neural network on MNIST.

    Args:
        epochs: Number of training epochs
        learning_rate: Learning rate for Adam optimizer
        n_layers: Number of variational layers in the quantum circuit
        entanglement_type: Entanglement pattern - 'linear', 'circular', or 'full'
        embedding_type: Data encoding strategy - 'angle' or 'amplitude'
    Returns:
        Training loss, accuracy, gate depth and parameter count
    """
    model     = HybridQNN(4, n_layers, entanglement_type, embedding_type)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        total_loss, correct, total = 0, 0, 0
        for X, y in loader:
            optimizer.zero_grad()
            out  = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
            correct    += (out.argmax(1) == y).sum().item()
            total      += len(y)

    acc        = correct / total
    cnot_count = {"linear": 3, "circular": 4, "full": 6}[entanglement_type]
    gate_depth = n_layers * (4 + cnot_count)
    param_count = n_layers * 4

    result = {
        "Architecture":  f"{n_layers}L-{entanglement_type}-{embedding_type}",
        "Accuracy":      round(acc, 4),
        "Gate Depth":    gate_depth,
        "Loss":          round(total_loss/len(loader), 4),
        "Acc/Depth Ratio": round(acc/gate_depth, 5)
    }
    search_results.append(result)

    return (
        f"loss={result['Loss']}, accuracy={result['Accuracy']}, "
        f"gate_depth={gate_depth}, params={param_count}, "
        f"architecture={result['Architecture']}"
    )

In [ ]:
@define_tool()
def describe_circuit(n_layers: int, entanglement_type: str, embedding_type: str) -> str:
    """Describes the structure of a quantum circuit architecture.

    Args:
        n_layers: Number of variational layers
        entanglement_type: Entanglement pattern - 'linear', 'circular', or 'full'
        embedding_type: Data encoding - 'angle' or 'amplitude'
    Returns:
        Human readable description of circuit structure including gate count and depth
    """
    n_qubits   = 4
    cnot_count = {"linear": 3, "circular": 4, "full": 6}[entanglement_type]
    gate_depth = n_layers * (n_qubits + cnot_count)
    param_count = n_layers * n_qubits
    entanglement_map = {
        "linear":   "0→1→2→3 (chain)",
        "circular": "0→1→2→3→0 (ring)",
        "full":     "all-to-all (complete graph)"
    }[entanglement_type]
    return f"""
Circuit Architecture: {n_layers}L-{entanglement_type}-{embedding_type}
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Qubits: {n_qubits} | Layers: {n_layers} | Embedding: {embedding_type}
Entanglement: {entanglement_map}
Gate depth: {gate_depth} | Parameters: {param_count}
"""

In [ ]:
search_results.clear()

agent3 = Agent(
    llm=VLLM(model='deepseek-chat', host='https://api.deepseek.com', api_key=api_key),
    tools=[describe_circuit, train_qnn],
    system_prompt="""You are a quantum circuit architecture optimization assistant.
First find the optimal learning rate using the baseline architecture (n_layers=2,
entanglement_type='linear', embedding_type='angle'). Try at least 5 learning rates.
Then use the best lr to explore at least 5 different architectures by varying
n_layers (1-4) and entanglement_type (linear, circular, full).
Use describe_circuit before each training run.
Report the best architecture ranked by accuracy-per-gate-depth ratio."""
)

response = agent3.run(
    "Find the optimal learning rate, then search for the best circuit architecture.",
    max_iterations=25
)
print(response.text)

In [ ]:
from IPython.display import display

df = pd.DataFrame(search_results)
if not df.empty:
    df = df.drop_duplicates(subset=["Architecture"])
    df = df.sort_values("Acc/Depth Ratio", ascending=False)
    display(df.style.highlight_max(subset=["Accuracy", "Acc/Depth Ratio"], color="lightgreen"))